In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters python-dotenv

In [1]:
# (이미 설치되어 있으면 다시 안 돌려도 됨)
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import create_retriever_tool
from langchain.agents import create_agent
from langchain.agents import AgentExecutor
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.embeddings import FastEmbedEmbeddings

embd = FastEmbedEmbeddings()

load_dotenv()  # .env 에서 OPENROUTER_API_KEY 사용

llm = ChatOpenAI(
    model="deepseek/deepseek-chat",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,  # 일부러 약간의 랜덤성 유지
)

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (/usr/local/python/3.12.1/lib/python3.12/site-packages/langchain/agents/__init__.py)

In [3]:
voc_text_bank = """
[KB은행 VOC & NPS 요약: 2025년 10월 vs 2025년 11월]

1. VOC 유형 중 '불만' 기준
- 지난달(2025-10) 불만 VOC: 1,200건
- 이번달(2025-11) 불만 VOC: 1,500건
- 지난달 → 이번달 불만 VOC 증감률: +25% (300건 증가)

2. NPS 점수
- 지난달(2025-10) NPS: 48점
- 이번달(2025-11) NPS: 44점
- 지난달 → 이번달 NPS 변화: -4포인트 (악화)

3. 해석
- 불만 VOC가 25%나 증가하면서, NPS도 4포인트 하락한 상황.
- 특히 모바일 앱 오류 및 창구 대기시간 관련 불만 비중이 높아진 것이 주요 원인으로 분석됨.
"""

voc_text_card = """
[KB카드 VOC & NPS 요약: 2025년 10월 vs 2025년 11월]

1. VOC 유형 중 '불만' 기준
- 지난달(2025-10) 불만 VOC: 800건
- 이번달(2025-11) 불만 VOC: 640건
- 지난달 → 이번달 불만 VOC 증감률: -20% (160건 감소)

2. NPS 점수
- 지난달(2025-10) NPS: 36점
- 이번달(2025-11) NPS: 40점
- 지난달 → 이번달 NPS 변화: +4포인트 (개선)

3. 해석
- 무이자 할부 프로모션 확대, 앱 결제 UX 개선으로 불만 VOC는 20% 감소.
- 동시에 추천 의향(NPS)이 4포인트 개선되어, 프로모션/UX 개선 효과가 반영된 것으로 해석 가능.
"""

voc_text_ins = """
[KB손해보험 VOC & NPS 요약: 2025년 10월 vs 2025년 11월]

1. VOC 유형 중 '불만' 기준
- 지난달(2025-10) 불만 VOC: 500건
- 이번달(2025-11) 불만 VOC: 525건
- 지난달 → 이번달 불만 VOC 증감률: +5% (25건 증가)

2. NPS 점수
- 지난달(2025-10) NPS: 30점
- 이번달(2025-11) NPS: 32점
- 지난달 → 이번달 NPS 변화: +2포인트 (소폭 개선)

3. 해석
- 자동차보험 사고처리 지연 VOC는 다소 늘었지만,
  장기보험 상품 리모델링과 상담품질 개선으로 전반적인 고객 추천 의향은 소폭 개선.
"""

voc_text_sec = """
[KB증권 VOC & NPS 요약: 2025년 10월 vs 2025년 11월]

1. VOC 유형 중 '불만' 기준
- 지난달(2025-10) 불만 VOC: 300건
- 이번달(2025-11) 불만 VOC: 330건
- 지난달 → 이번달 불만 VOC 증감률: +10% (30건 증가)

2. NPS 점수
- 지난달(2025-10) NPS: 28점
- 이번달(2025-11) NPS: 26점
- 지난달 → 이번달 NPS 변화: -2포인트 (악화)

3. 해석
- 해외주식 주문 지연, MTS 장애 관련 불만 증가로
  단기적으로 고객 체감 만족도가 악화된 상황.
"""

voc_text_life = """
[KB라이프 VOC & NPS 요약: 2025년 10월 vs 2025년 11월]

1. VOC 유형 중 '불만' 기준
- 지난달(2025-10) 불만 VOC: 200건
- 이번달(2025-11) 불만 VOC: 220건
- 지난달 → 이번달 불만 VOC 증감률: +10% (20건 증가)

2. NPS 점수
- 지난달(2025-10) NPS: 35점
- 이번달(2025-11) NPS: 34점
- 지난달 → 이번달 NPS 변화: -1포인트 (소폭 악화)

3. 해석
- 보험금 지급 심사기간에 대한 VOC가 일부 늘면서,
  NPS가 소폭 하락한 것으로 해석 가능.
"""

company_texts = {
    "bank":  voc_text_bank,
    "card":  voc_text_card,
    "ins":   voc_text_ins,
    "sec":   voc_text_sec,
    "life":  voc_text_life,
}


In [4]:
def build_voc_retriever(company_key: str, text: str):
    docs = [Document(page_content=text, metadata={"company": company_key})]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=0,
    )

    splits = text_splitter.split_documents(docs)

    # 🔥 API Key 필요 없음 (Colab에서 가장 안정적)
    embd = FastEmbedEmbeddings()

    persist_dir = f"db_voc_{company_key}"

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embd,
        persist_directory=persist_dir,
    )

    return vectorstore.as_retriever()

retriever_bank = build_voc_retriever("bank", company_texts["bank"])
retriever_card = build_voc_retriever("card", company_texts["card"])
retriever_ins  = build_voc_retriever("ins",  company_texts["ins"])
retriever_sec  = build_voc_retriever("sec",  company_texts["sec"])
retriever_life = build_voc_retriever("life", company_texts["life"])


In [5]:
bank_tool = create_retriever_tool(
    retriever=retriever_bank,
    name="bank_voc",
    description="KB은행의 지난달/이번달 VOC 및 NPS 요약 정보를 조회할 때 사용합니다.",
)

card_tool = create_retriever_tool(
    retriever=retriever_card,
    name="card_voc",
    description="KB카드의 지난달/이번달 VOC 및 NPS 요약 정보를 조회할 때 사용합니다.",
)

ins_tool = create_retriever_tool(
    retriever=retriever_ins,
    name="ins_voc",
    description="KB손해보험의 지난달/이번달 VOC 및 NPS 요약 정보를 조회할 때 사용합니다.",
)

sec_tool = create_retriever_tool(
    retriever=retriever_sec,
    name="sec_voc",
    description="KB증권의 지난달/이번달 VOC 및 NPS 요약 정보를 조회할 때 사용합니다.",
)

life_tool = create_retriever_tool(
    retriever=retriever_life,
    name="life_voc",
    description="KB라이프의 지난달/이번달 VOC 및 NPS 요약 정보를 조회할 때 사용합니다.",
)

tools = [bank_tool, card_tool, ins_tool, sec_tool, life_tool]
tools

[Tool(name='bank_voc', description='KB은행의 지난달/이번달 VOC 및 NPS 요약 정보를 조회할 때 사용합니다.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x7dd554a8e5c0>, retriever=VectorStoreRetriever(tags=['Chroma', 'FastEmbedEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7dd543b00620>, search_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_separator='\n\n', response_format='content'), coroutine=functools.partial(<function _aget_relevant_documents at 0x7dd554a7d3a0>, retriever=VectorStoreRetriever(tags=['Chroma', 'FastEmbedEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7dd543b00620>, search_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_separator

In [6]:
template = """
당신은 KB금융그룹의 VOC/NPS 리포트를 읽고 설명해주는 데이터 분석 에이전트입니다.
아래 도구들을 통해 계열사별 지난달 vs 이번달 VOC·NPS 요약을 조회할 수 있습니다.

사용 가능한 도구:
{tools}

[도구 선택 규칙]
- 질문에 '은행' 이라는 단어가 포함되면 반드시 bank_voc 도구만 사용해서 먼저 데이터를 조회하십시오.
- 질문에 '카드' 이라는 단어가 포함되면 반드시 card_voc 도구만 사용해서 먼저 데이터를 조회하십시오.
- 질문에 '손보' 또는 '손해보험' 이라는 단어가 포함되면 ins_voc 도구를 사용하십시오.
- 질문에 '증권' 이라는 단어가 포함되면 sec_voc 도구를 사용하십시오.
- 질문에 '라이프' 또는 '생명' 이라는 단어가 포함되면 life_voc 도구를 사용하십시오.
- 한 질문에 여러 계열사가 함께 나오면, 필요한 만큼 여러 도구를 순차적으로 호출해도 됩니다.

[출력 시 유의사항]
- 반드시 도구를 사용해서 Observation(관측 결과)을 얻은 뒤에 최종 답변을 작성하십시오.
- 질문이 요구하는 지표:
  - 지난달 vs 이번달 '불만 VOC' 증감률 (예: +25%, -10% 등)
  - 지난달 vs 이번달 NPS 점수 증감 (예: +4포인트, -3포인트 등)
- 최종 답변에는 수치(지난달 값, 이번달 값, 증감률/증감포인트)를 모두 포함하고,
  마지막에 한 줄 요약 인사이트도 제공하십시오.

[대화 형식]
아래 형식을 엄격하게 따르십시오.

Question: 사용자가 물어본 질문
Thought: 무엇을 할지 먼저 간단히 생각합니다.
Action: 사용할 도구 이름. 반드시 [{tool_names}] 중 하나여야 합니다.
Action Input: 도구에 전달할 구체적인 검색 질의(한국어로 작성)
Observation: 도구를 실행한 뒤 얻은 결과

... (Thought / Action / Action Input / Observation 사이클은 필요할 때까지 반복 가능)

Thought: 이제 최종 답변을 알겠습니다
Final Answer: 사용자의 질문에 대한 최종 한국어 답변. 표나 bullet를 활용해 깔끔하게 정리하십시오.

반드시 Thought -> Action -> Action Input -> Observation 순서를 지키십시오.
Action 없이 Observation을 만들지 마십시오.

이제 시작합니다.

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)


In [15]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="(여기에 시스템 프롬프트 문자열)"
)

react_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)


TypeError: functools.partial(<function _get_relevant_documents at 0x7dd554a8e5c0>, retriever=VectorStoreRetriever(tags=['Chroma', 'FastEmbedEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7dd543b00620>, search_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_separator='\n\n', response_format='content') is not a module, class, method, or function.